In [2]:
import os
import numpy as np
import librosa
import glob
import plotly.express as px
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tqdm import tqdm


train_dir = './audios/rec/fft/train/'
test_dir = './audios/rec/fft/test'
num_files_to_copy = int(len(os.listdir(train_dir))*0.1)
anomaly_files = glob.glob(os.path.join(train_dir, 'anomaly*'))
contamination = len(anomaly_files) / len(os.listdir(train_dir))

print(len(os.listdir(train_dir)), "10%:", num_files_to_copy, "anomalies:", len(anomaly_files), "contamination:", contamination)
print(len(os.listdir(test_dir)))

2805 10%: 280 anomalies: 276 contamination: 0.09839572192513368
2573


In [3]:
pos = glob.glob(os.path.join("./another_tutorial/audio_segments_test_centrum_reduced/positive", '*.wav'))
neg = glob.glob(os.path.join("./another_tutorial/audio_segments_test_centrum_reduced/negative", '*.wav'))
print(len(pos), pos[:3])
print(len(neg), neg[:3])
len(pos+neg)

2529 ['./another_tutorial/audio_segments_test_centrum_reduced/positive\\rec55_250_10_chunk_1.wav', './another_tutorial/audio_segments_test_centrum_reduced/positive\\rec55_250_10_chunk_10.wav', './another_tutorial/audio_segments_test_centrum_reduced/positive\\rec55_250_10_chunk_100.wav']
2573 ['./another_tutorial/audio_segments_test_centrum_reduced/negative\\rec43_250_10_chunk_1.wav', './another_tutorial/audio_segments_test_centrum_reduced/negative\\rec43_250_10_chunk_10.wav', './another_tutorial/audio_segments_test_centrum_reduced/negative\\rec43_250_10_chunk_100.wav']


5102

In [4]:
def extract_audio_features(audio_file):
    x, sr = librosa.load(audio_file)

    # Calculate the FFT
    fft_values = np.fft.rfft(x)
    magnitude = np.abs(fft_values)[:(len(x) + 1) // 2]  # Magnitude of the FFT

    return magnitude[:150]

In [5]:
# Load normal audio files
features = np.array([extract_audio_features(audio_file) for audio_file in pos+neg])
labels = np.array([0 if "negative" in audio_file else 1 for audio_file in pos+neg])
print(len(features), len(labels))
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

print(len(X_train), len(y_train), len(X_test), len(y_test))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=150),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(150, activation='linear')
])


model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, y_train, epochs=10)  # Train on normal data

5102 5102
4081 4081 1021 1021
Epoch 1/10
128/128 [==============================] - 1s 1ms/step - loss: 0.3991
Epoch 2/10
128/128 [==============================] - 0s 937us/step - loss: 0.0510
Epoch 3/10
128/128 [==============================] - 0s 910us/step - loss: 0.0323
Epoch 4/10
128/128 [==============================] - 0s 935us/step - loss: 0.0262
Epoch 5/10
128/128 [==============================] - 0s 987us/step - loss: 0.0225
Epoch 6/10
128/128 [==============================] - 0s 965us/step - loss: 0.0187
Epoch 7/10
128/128 [==============================] - 0s 943us/step - loss: 0.0174
Epoch 8/10
128/128 [==============================] - 0s 912us/step - loss: 0.0146
Epoch 9/10
128/128 [==============================] - 0s 917us/step - loss: 0.0125
Epoch 10/10
128/128 [==============================] - 0s 971us/step - loss: 0.0102


In [6]:
# Define a threshold for anomaly detection
threshold = 3.1  # You should adjust this threshold based on your data and problem.

# Initialize lists to store results
anomalies = []
normal_data = []

# Loop through your data and make predictions
for f in tqdm(neg, desc="Processing data"):
    features = extract_audio_features(f)
    features = features.reshape(1, -1)
    predictions = model.predict(features, verbose=None)

    # Calculate the reconstruction error (for anomaly detection)
    reconstruction_error = np.mean(np.square(features - predictions))

    # Check if the error is above the threshold
    if reconstruction_error < threshold:
        anomalies.append((f, reconstruction_error))
    else:
        normal_data.append((f, reconstruction_error))

Processing data:   1%|          | 24/2573 [00:01<02:05, 20.26it/s]


KeyboardInterrupt: 

In [7]:
# Print anomalies
print("anomalies:", len(anomalies), min(anomalies, key=lambda x: x[1]), max(anomalies, key=lambda x: x[1]))
for anomaly in anomalies:
    print(round(anomaly[1], 2), 0 if "negative" in anomaly[0] else 1, anomaly[0])

anomalies: 24 ('./another_tutorial/audio_segments_test_centrum_reduced/negative\\rec43_250_10_chunk_10.wav', 0.21346674883182734) ('./another_tutorial/audio_segments_test_centrum_reduced/negative\\rec43_250_10_chunk_113.wav', 0.5179912014471986)
0.25 0 ./another_tutorial/audio_segments_test_centrum_reduced/negative\rec43_250_10_chunk_1.wav
0.21 0 ./another_tutorial/audio_segments_test_centrum_reduced/negative\rec43_250_10_chunk_10.wav
0.27 0 ./another_tutorial/audio_segments_test_centrum_reduced/negative\rec43_250_10_chunk_100.wav
0.32 0 ./another_tutorial/audio_segments_test_centrum_reduced/negative\rec43_250_10_chunk_101.wav
0.26 0 ./another_tutorial/audio_segments_test_centrum_reduced/negative\rec43_250_10_chunk_102.wav
0.25 0 ./another_tutorial/audio_segments_test_centrum_reduced/negative\rec43_250_10_chunk_103.wav
0.28 0 ./another_tutorial/audio_segments_test_centrum_reduced/negative\rec43_250_10_chunk_104.wav
0.34 0 ./another_tutorial/audio_segments_test_centrum_reduced/negative\

In [8]:
# Print normal data
print("normal_data:", len(normal_data), min(normal_data, key=lambda x: x[1]), max(normal_data, key=lambda x: x[1]))
for normal in normal_data:
    print(round(normal[1], 2), 0 if "negative" in normal[0] else 1, normal[0])

ValueError: min() arg is an empty sequence

In [9]:
anomalies = []
normal_data = []
for f in tqdm(pos[:5]+neg[100:105], desc="Processing data"):
    features = extract_audio_features(f)
    features = features.reshape(1, -1)
    predictions = model.predict(features, verbose=None)
    
    reconstruction_error = np.mean(np.square(features - predictions))

    # Check if the error is above the threshold
    if reconstruction_error < threshold:
        anomalies.append((f, reconstruction_error))
    else:
        normal_data.append((f, reconstruction_error))

Processing data: 100%|██████████| 10/10 [00:00<00:00, 16.65it/s]


In [10]:
anomalies = [(" Ventil Nummer:" + filename[83:-4] + f" ({filename[55:62]})", error) for filename, error in anomalies]
normal_data = [(" Ventil Nummer:" + filename[83:-4] + f" ({filename[55:62]})", error) for filename, error in normal_data]

# Combine anomalies and normal data into a single list
combined_data = anomalies + normal_data

# Create a DataFrame from the combined data
import pandas as pd
df = pd.DataFrame(combined_data, columns=["File", "Reconstruction Error"])
df["Label"] = ["Anomaly" if error < threshold else "Normal" for error in df["Reconstruction Error"]]

# Create a scatter plot with color-coding
fig = px.bar(df, x="File", y="Reconstruction Error", color="Label", title="Anomalies and Normal Data")
fig.update_xaxes(categoryorder='total ascending')

# Add a horizontal line at y = -3
fig.add_hline(y=3, line_dash="dash", line_color="red", annotation_text="Threshold", annotation_position="bottom right")

# Display the chart
fig.show()